# Transformations: Normalize, Enrich, Derive, Windows, SCD2

## What you'll practice

Transformations turn valid data into useful data: normalization, lookups (joins), derived columns, window functions and slowly changing dimensions.
Write them as **pure functions** (input → new output, no mutation) — the exact discipline Spark's immutable DataFrames enforce.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/etl/test_08_03_transformations.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/08_data_engineering_python/03_transformations.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Normalize customer records

**Implement:** `normalize_customer(row: dict) -> dict`

From a raw `customers.csv` row return a new dict: `name` stripped and title-cased (collapse inner whitespace), `email` stripped+lowercased (blank → `None`),
`country` upper, `signup_date` as `YYYY-MM-DD` accepting `YYYY-MM-DD` or `DD/MM/YYYY` (else `None`), `is_active` as bool. Other fields copied unchanged.

**Input:** `row: dict`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_normalize_customer` in `tests/etl/test_08_03_transformations.py`.

**Check:** `pytest tests/etl/test_08_03_transformations.py -k test_normalize_customer`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Enrich via lookups (hash joins)

**Implement:** `enrich_orders(orders: list[dict], customers: list[dict], products: list[dict]) -> list[dict]`

Add `customer_segment`, `customer_country` (from customers by `customer_id`) and `product_category`, `product_name` (from products by `product_id`) to each order. Missing matches → `None` (a left join). Build each lookup dict once.

**Input:** `orders: list[dict], customers: list[dict], products: list[dict]`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_enrich_orders` in `tests/etl/test_08_03_transformations.py`.

**Check:** `pytest tests/etl/test_08_03_transformations.py -k test_enrich_orders`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Derived columns

**Implement:** `derive(order: dict) -> dict`

Given a typed order with `order_ts` (aware UTC datetime) and `amount`, add: `order_date` (UTC date ISO), `order_hour_ist` (hour in Asia/Kolkata, use `zoneinfo`),
`weekday` (e.g. `"Monday"`, in IST), `is_weekend` (IST), `amount_bucket` (`"<1k"`, `"1k-10k"` for 1000 ≤ amount < 10000, `"10k+"`).

**Input:** `order: dict`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_derive` in `tests/etl/test_08_03_transformations.py`.

**Check:** `pytest tests/etl/test_08_03_transformations.py -k test_derive`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Window functions in Python

**Implement:** `add_window_columns(rows: list[dict], partition_by: str, order_by: str, value: str) -> list[dict]`

Emulate SQL `ROW_NUMBER()`, `SUM() OVER (... ROWS UNBOUNDED PRECEDING)` and `LAG()`: for each partition sorted by `order_by`, add `row_number` (1-based), `running_total`
(rounded 2) and `prev_value` (`None` for the first row). Output ordered by partition (sorted) then `order_by`. Ties in `order_by` keep input order.

**Input:** `rows: list[dict], partition_by: str, order_by: str, value: str`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
add_window_columns([], 'a', 'b', 'c')
# -> []
```

**Check:** `pytest tests/etl/test_08_03_transformations.py -k test_add_window_columns`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Slowly Changing Dimension Type 2 · 🧩 Mini-project

**Implement:** `apply_scd2(dimension: list[dict], updates: list[dict], key: str, tracked: list[str], effective_date: str) -> list[dict]`

The dimension holds versions: `{key, ...attributes, "valid_from", "valid_to" (None if current), "is_current"}`. For each update row (latest attributes for a key):
- **new key** → insert a current version (`valid_from = effective_date`);
- **tracked attribute changed** vs the current version → close the current version (`valid_to = effective_date`, `is_current = False`) and insert a new current version;
- **only untracked attributes changed** → update them in place on the current version (Type 1 behaviour);
- **unchanged** → nothing. Keys absent from `updates` are left alone (no deletes).
Duplicate keys within `updates` → `ValueError`. Return a new list sorted by key then `valid_from`; never mutate inputs. Running the same updates twice must be a no-op the second time.

**Input:** `dimension: list[dict], updates: list[dict], key: str, tracked: list[str], effective_date: str`

**Expected output:** `list[dict]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_apply_scd2, test_apply_scd2_duplicates` in `tests/etl/test_08_03_transformations.py`.

**Check:** `pytest tests/etl/test_08_03_transformations.py -k "test_apply_scd2 or test_apply_scd2_duplicates"`

**YOUR CODE ↓**